# Process games on Google Colab

Runs the Hoops Tracker pipeline (players, teams, jersey numbers, positions, stats) on Colab's free
graphics card instead of the laptop. It works through the games in your Google Drive folder and
puts the results back there.

**Before you start** (once): in Google Drive make the folder named below, with `models/` holding
`player_detector.pt` from the laptop's models folder. For each game: put the video in `inbox/`,
and the court files for it (from the laptop's `data/<game>/` folder: `court_setup.json`,
`court_calibration.json`, `hoops.json` and the `court_views` folder) in `inbox/<game>/`.
The dashboard can do this for you.

Then **Runtime > Change runtime type > T4 GPU**, and **Runtime > Run all**. Each cell below is one step.

## 1. Settings: the name of your Drive folder

In [3]:
DRIVE_FOLDER_NAME = "Hoops Tracker"     # the folder in "My Drive" with models/, inbox/ and outbox/

## 2. Connect Google Drive (a window asks you to allow it)

In [4]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
print("Drive folder:", DRIVE_ROOT)

Mounted at /content/drive
Drive folder: /content/drive/MyDrive/Hoops Tracker


## 3. Get the program from GitHub (the latest version of the code only)

In [5]:
%cd /content
!git clone --depth 1 https://github.com/gsharp23/basketball-tracker.git || (cd basketball-tracker && git pull)
%cd /content/basketball-tracker

/content
Cloning into 'basketball-tracker'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 107 (delta 2), reused 53 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 458.42 KiB | 9.96 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/basketball-tracker


## 4. Install the pieces Colab does not have (torch is already here, with the graphics card support)

In [6]:
%pip install -q ultralytics==8.4.172 easyocr==1.7.2 supervision==0.30.6 imageio-ffmpeg==0.6.0

## 5. Check that the graphics card is there (if it says False, do Runtime > Change runtime type > T4 GPU)

In [7]:
import torch

print("Graphics card available:", torch.cuda.is_available())
!nvidia-smi -L

Graphics card available: True
GPU 0: Tesla T4 (UUID: GPU-29b2167a-51d0-ce78-9707-bec5efc53230)


## 6. Process the games. This cell keeps running: it looks in the inbox every minute and processes what it finds

In [ ]:
!python src/colab_worker.py --drive-root "{DRIVE_ROOT}"

# The worker stops by itself after 15 minutes with nothing to do. Then this machine is let go of, so no
# more compute units are used; the next Run all gets a fresh one and carries on where things stopped.
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as why:   # not on Colab, or Colab said no: then stop it by hand
    print("Could not release the machine by itself:", why, "- use Runtime > Disconnect and delete runtime.")

2026-11-26 16:14:50  Worker started. Drive folder: /content/drive/MyDrive/Hoops Tracker
2026-11-26 16:14:52  --- Crossroads-Christian_2026-11-17.mp4: starting
2026-11-26 16:15:26  Copying Crossroads-Christian_2026-11-17.mp4 to the clips folder (1532 MB)...
2026-11-26 16:15:52  Running: /usr/bin/python3 /content/basketball-tracker/src/main.py --video clips/Crossroads-Christian_2026-11-17.mp4
Running the pipeline on clips/Crossroads-Christian_2026-11-17.mp4
NOTE: the court has not been marked for this video yet. Detection, teams and jersey numbers run
      first. You can mark the court meanwhile, in a second terminal. In that terminal first run
        cd /content/basketball-tracker
        source ~/venvs/hoops-tracker/bin/activate
      and then:
        The reference pictures of the court have not been chosen. Run:  python src/select_court.py --video clips/Crossroads-Christian_2026-11-17.mp4
        The court landmarks have not been clicked. Run:  python src/select_court_keypoints.py 

## 7. Stopping

- The cell above runs until you stop it: press its stop button (or **Runtime > Interrupt execution**).
  Nothing is lost: the results so far are in `outbox/<game>/` in your Drive, and the next run carries
  on where it stopped.
- **Closing this tab, or leaving it idle, stops the processing.** A free Colab session also ends by
  itself after about 12 hours. Open the notebook and run all again; finished stages are skipped.
- Nothing happens unless this notebook is running.
- What comes back: `outbox/<game>/status.json` (done, failed, needs_court or running),
  `data/` (stats and every result file), `outputs/` (the videos and pictures) and `crops/`.
  `needs_court` means the players were found but the court files are missing: mark the court on the
  laptop, put the files in `inbox/<game>/`, and the rest is done the next time the worker looks.